# Nigeria Topic-Relevance Pipeline (v2 — binary, no labeling)

1. Keyword pre-filter over all comments (keeps `Highly relevant` + `Moderately relevant` keywords)
2. For every keyword-hit comment, ask `gpt-4o-mini` a **binary** yes/no on topic relevance
3. Save only the `yes` comments — no themes, no sentiment, no threshold

Output: `Topic Relevant Comments - Nigeria - Final/<creator>/<video>.xlsx` + `_summary.xlsx`.


In [1]:
from __future__ import annotations

import asyncio
import json
import os
import re
import time
import unicodedata
from pathlib import Path
from typing import Any

import pandas as pd
from dotenv import load_dotenv
from openai import AsyncOpenAI
from tenacity import retry, stop_after_attempt, wait_exponential_jitter
from tqdm.asyncio import tqdm as atqdm
from tqdm.auto import tqdm


def find_root(start: Path) -> Path:
    start = start.resolve()
    for c in [start, *start.parents]:
        if (c / "Nigeria Audience Comments").exists() and (c / "Codebook and Keywords").exists():
            return c
    return start


ROOT = find_root(Path.cwd())
load_dotenv(ROOT / ".env")

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "").strip()
if not OPENAI_API_KEY:
    raise RuntimeError("OPENAI_API_KEY missing in .env")

INPUT_GLOB = "Nigeria Audience Comments/**/*.xlsx"
KEYWORDS_XLSX = ROOT / "Codebook and Keywords" / "NLC Proposed keywords.xlsx"
KEYWORDS_SHEET = "Nigeria"
KEYWORD_COL = "Keyword"
KEYWORD_RELEVANCE_COL = "Relevance to manosphere conversations"
KEYWORD_LEVELS_TO_KEEP = {"Highly relevant", "Moderately relevant"}

OUTPUT_DIR = ROOT / "Topic Relevant Comments - Nigeria - Final"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "gpt-4o-mini"
CONCURRENCY = 12
REQUEST_TIMEOUT = 30
DRY_RUN_ROWS = None  # set to e.g. 30 for a smoke test

async_client = AsyncOpenAI(api_key=OPENAI_API_KEY, timeout=REQUEST_TIMEOUT)
print("Root:", ROOT)
print("Output:", OUTPUT_DIR)
print("Model:", MODEL_NAME, "| concurrency:", CONCURRENCY)


Root: /Users/sushildalavi/Desktop/NLC/GATES-Project
Output: /Users/sushildalavi/Desktop/NLC/GATES-Project/Topic Relevant Comments - Nigeria - Final
Model: gpt-4o-mini | concurrency: 12


In [2]:
def normalize_text(v: Any) -> str:
    if pd.isna(v):
        return ""
    t = unicodedata.normalize("NFKC", str(v))
    t = t.replace("\u2018", "'").replace("\u2019", "'")
    t = t.replace("\u201c", '"').replace("\u201d", '"')
    return re.sub(r"\s+", " ", t).strip()


def load_keywords() -> pd.DataFrame:
    df = pd.read_excel(KEYWORDS_XLSX, sheet_name=KEYWORDS_SHEET)
    df.columns = [str(c).strip() for c in df.columns]
    kw = df[[KEYWORD_COL, KEYWORD_RELEVANCE_COL]].copy()
    kw.columns = ["keyword", "relevance"]
    kw["keyword"] = kw["keyword"].map(normalize_text)
    kw["relevance"] = kw["relevance"].map(normalize_text)
    kw = kw[(kw["keyword"] != "") & (kw["relevance"].isin(KEYWORD_LEVELS_TO_KEEP))]
    return kw.drop_duplicates("keyword").reset_index(drop=True)


def compile_patterns(keywords: list[str]) -> list[tuple[str, re.Pattern[str]]]:
    pats: list[tuple[str, re.Pattern[str]]] = []
    for kw in keywords:
        base = normalize_text(kw).lower().lstrip("#")
        if not base:
            continue
        escaped = re.escape(base).replace(r"\ ", r"\s+")
        pats.append((kw, re.compile(rf"(?<!\w)#?{escaped}(?!\w)", re.IGNORECASE)))
    return pats


def find_text_col(cols: list[str]) -> str:
    low = {str(c).strip().lower(): c for c in cols}
    for k in ["text", "comment", "comments", "content", "comment text"]:
        if k in low:
            return low[k]
    raise ValueError(f"No comment column in {cols}")


def keyword_hits(text: str, patterns: list[tuple[str, re.Pattern[str]]]) -> list[str]:
    return [kw for kw, pat in patterns if pat.search(text)]


In [3]:
kw_df = load_keywords()
patterns = compile_patterns(kw_df["keyword"].tolist())
print(f"Loaded {len(kw_df)} keywords")

files = sorted(ROOT.glob(INPUT_GLOB))
files = [f for f in files if f.is_file() and not f.name.startswith("~$")]
print(f"Found {len(files)} input files")

rows = []
ORIGINAL_COLUMNS: dict[str, list[str]] = {}
for path in files:
    raw = pd.read_excel(path)
    raw.columns = [str(c).strip() for c in raw.columns]
    tcol = find_text_col(list(raw.columns))
    src = path.relative_to(ROOT).as_posix()
    ORIGINAL_COLUMNS[src] = list(raw.columns)
    df = raw.copy().reset_index(drop=True)
    df["__creator"] = path.parent.name
    df["__video"] = path.stem
    df["__source_file"] = src
    df["__comment_text"] = df[tcol].map(normalize_text)
    df = df[df["__comment_text"] != ""].copy()
    rows.append(df)

master = pd.concat(rows, ignore_index=True)
print(f"Total non-empty comments: {len(master)}")

master["__kw_hits"] = master["__comment_text"].map(lambda t: keyword_hits(t, patterns))
hits = master[master["__kw_hits"].map(len) > 0].reset_index(drop=True).copy()
print(f"Keyword-hit comments to classify: {len(hits)}")

if DRY_RUN_ROWS:
    hits = hits.head(DRY_RUN_ROWS).copy()
    print(f"DRY RUN: classifying only {len(hits)} rows")


Loaded 332 keywords
Found 10 input files


Total non-empty comments: 9735


Keyword-hit comments to classify: 2840


In [4]:
SYSTEM_PROMPT = """You are a topic-relevance classifier for manosphere / gender-norms discourse in Nigerian YouTube audience comments.

A comment is RELEVANT if it engages in any way — directly or indirectly — with topics such as: gender roles, marriage, dating, relationships, sexuality, women's agency or independence, fatherhood, motherhood, masculinity or femininity, patriarchy, provider norms, traditional vs modern values, feminism, "alpha/beta" / red-pill ideology, male/female power dynamics, gender-based expectations or violence, or related manosphere themes.

Be generous. If the comment takes any stance, shares any personal experience, reacts emotionally to the topic, or adds any opinion, nuance, or anecdote connected to these themes, mark it RELEVANT. When in doubt, lean YES.

Reply with JSON ONLY. You MUST pick yes or no:
{"relevant": "yes" | "no", "reason": "<one short sentence>"}"""


def build_user_prompt(comment: str, kw_hits: list[str]) -> str:
    return (
        f"Keyword hits: {kw_hits}\n\n"
        f"Comment:\n\"\"\"{comment}\"\"\"\n\n"
        "Return JSON: {\"relevant\": \"yes\"|\"no\", \"reason\": \"...\"}."
    )


@retry(stop=stop_after_attempt(3), wait=wait_exponential_jitter(initial=1, max=8))
async def classify_one(sem: asyncio.Semaphore, comment: str, kw_hits: list[str]) -> tuple[str, int, int]:
    async with sem:
        r = await async_client.chat.completions.create(
            model=MODEL_NAME,
            temperature=0,
            response_format={"type": "json_object"},
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": build_user_prompt(comment, kw_hits)},
            ],
        )
        parsed = json.loads(r.choices[0].message.content)
        rel = str(parsed.get("relevant", "")).strip().lower()
        if rel not in {"yes", "no"}:
            rel = "no"
        pt = getattr(r.usage, "prompt_tokens", 0) or 0
        ct = getattr(r.usage, "completion_tokens", 0) or 0
        return rel, pt, ct


async def classify_all_ordered(df: pd.DataFrame) -> tuple[list[str], int, int]:
    sem = asyncio.Semaphore(CONCURRENCY)

    async def _one(idx: int, comment: str, kw_hits: list[str]):
        try:
            rel, pt, ct = await classify_one(sem, comment, kw_hits)
        except Exception:
            rel, pt, ct = "no", 0, 0
        return idx, rel, pt, ct

    tasks = [
        _one(i, row["__comment_text"], row["__kw_hits"])
        for i, (_, row) in enumerate(df.iterrows())
    ]
    results: list[str] = [""] * len(tasks)
    pin = 0
    pout = 0
    for coro in atqdm.as_completed(tasks, total=len(tasks), desc="Classifying"):
        idx, rel, pt, ct = await coro
        results[idx] = rel
        pin += pt
        pout += ct
    return results, pin, pout


t0 = time.time()
relevant_flags, pin_total, pout_total = await classify_all_ordered(hits)
elapsed = time.time() - t0

hits["__relevant"] = relevant_flags
kept = hits[hits["__relevant"] == "yes"].copy()
print(f"Classified: {len(hits)} in {elapsed:.1f}s | Kept (yes): {len(kept)} | Rejected (no): {(hits['__relevant']=='no').sum()}")

cost = (pin_total / 1_000_000) * 0.15 + (pout_total / 1_000_000) * 0.60
print(f"Tokens in: {pin_total:,} | out: {pout_total:,} | est cost: ${cost:.4f}")


Classifying:   0%|          | 0/2840 [00:00<?, ?it/s]

Classifying:   0%|          | 1/2840 [00:00<38:24,  1.23it/s]

Classifying:   0%|          | 10/2840 [00:01<03:45, 12.54it/s]

Classifying:   0%|          | 13/2840 [00:01<04:37, 10.18it/s]

Classifying:   1%|          | 20/2840 [00:01<02:54, 16.18it/s]

...

Classifying: 100%|█████████▉| 2838/2840 [2:46:34<00:01,  1.02it/s]

Classifying: 100%|█████████▉| 2839/2840 [2:46:35<00:00,  1.11it/s]

Classifying: 100%|██████████| 2840/2840 [2:46:42<00:00,  2.36s/it]

Classifying: 100%|██████████| 2840/2840 [2:46:42<00:00,  3.52s/it]

Classified: 2840 in 10002.8s | Kept (yes): 509 | Rejected (no): 2331
Tokens in: 321,164 | out: 33,007 | est cost: $0.0680


In [5]:
summary_rows = []
for (creator, video, src), grp in kept.groupby(["__creator", "__video", "__source_file"]):
    out_dir = OUTPUT_DIR / creator
    out_dir.mkdir(parents=True, exist_ok=True)
    orig_cols = ORIGINAL_COLUMNS.get(src, [c for c in grp.columns if not c.startswith("__")])
    grp[orig_cols].to_excel(out_dir / f"{video}.xlsx", index=False)
    summary_rows.append({"creator": creator, "video": video, "kept": len(grp)})

total_kw_hits_per_video = (
    hits.groupby(["__creator", "__video"]).size().rename("total_kw_hits").reset_index()
)
summary = pd.DataFrame(summary_rows).merge(
    total_kw_hits_per_video.rename(columns={"__creator": "creator", "__video": "video"}),
    on=["creator", "video"], how="outer"
).fillna(0)
summary[["kept", "total_kw_hits"]] = summary[["kept", "total_kw_hits"]].astype(int)
summary.to_excel(OUTPUT_DIR / "_summary.xlsx", index=False)
print(f"Wrote kept comments to {OUTPUT_DIR}")
print(summary.to_string(index=False))


Wrote kept comments to /Users/sushildalavi/Desktop/NLC/GATES-Project/Topic Relevant Comments - Nigeria - Final
         creator                                        video  kept  total_kw_hits
   Agba John Doe Never Leave Marriage Because Husband Cheated   103            276
   Agba John Doe              Single Woman Earning Above 1.5M    37            102
Banky Wellington                              Final Say Faith   166           1664
Banky Wellington      My Story Journey Through Hope and Faith     7            156
Deyemi Okanlawon                Men Must Hold Men Accountable    10             22
Deyemi Okanlawon                   Stop Raping Women Response    64            234
           Shola            7 Women Will Beg One Man to Marry    44            142
           Shola                  Women and Availability Trap    16             63
         Wizarab                    Child Support and Divorce    25             62
         Wizarab               Sex Toys and Raping Young Bo